# Bias Annotation — Dinithi (750 articles)

## Instructions
1. Run all cells below to load your dataset and the annotation widget.
2. For each article, read the **title** and **body text** carefully.
3. Select a **bias label** from the dropdown based on the guidelines below.
4. Click **Save & Next** to record your label and move to the next article.
5. Your progress is **auto-saved** to `annotations/dinithi.csv` after every label.
6. You can close and resume anytime — it picks up where you left off.

## Bias Buckets (assign ONE per article)

| Label | Description |
|-------|-------------|
| `far_left` | Strong left-leaning political framing |
| `left` | Moderate left-leaning tone or source selection |
| `center` | Balanced, neutral reporting |
| `right` | Moderate right-leaning tone or source selection |
| `far_right` | Strong right-leaning political framing |

## Tips
- Focus on **how** the story is told (framing, word choice, source selection), not **what** the story is about.
- If unsure between two labels, pick the one that feels stronger.
- Flag articles that are unclear or don't fit any category using the **Flag** button.

In [ ]:
import pandas as pd
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

ANNOTATOR = "dinithi"
CSV_PATH = f"../annotations/{ANNOTATOR}.csv"
BIAS_LABELS = ["far_left", "left", "center", "right", "far_right"]

df = pd.read_csv(CSV_PATH)
df["bias_label"] = df["bias_label"].fillna("").astype(str)

# Find first unlabeled article
current_idx = df[df["bias_label"] == ""].index[0] if (df["bias_label"] == "").any() else len(df)
labeled_count = (df["bias_label"] != "").sum()
print(f"Loaded {len(df)} articles. {labeled_count} already labeled. Starting at article {current_idx + 1}.")

In [ ]:
def save_csv():
    df.to_csv(CSV_PATH, index=False)

def show_article(idx):
    clear_output(wait=True)
    labeled = (df["bias_label"] != "").sum()
    
    if idx >= len(df):
        print(f"All {len(df)} articles annotated! You're done.")
        save_csv()
        return

    row = df.iloc[idx]
    
    display(HTML(f"""
    <div style="border:1px solid #ccc; padding:15px; margin:10px 0; border-radius:8px; max-width:900px;">
        <h3>Article {idx + 1} / {len(df)} &nbsp; | &nbsp; Progress: {labeled} labeled</h3>
        <p><b>Publisher:</b> {row['publisher']} &nbsp;&nbsp; <b>Date:</b> {row['published_at']}</p>
        <p><b>Article ID:</b> <code>{row['article_id']}</code></p>
        <hr>
        <h3>{row['title']}</h3>
        <div style="white-space:pre-wrap; line-height:1.8; font-size:14px; max-height:500px; overflow-y:auto; padding:10px; background:#f9f9f9; border-radius:4px;">
{row['body_text']}
        </div>
    </div>
    """))

    # Label dropdown
    dropdown = widgets.Dropdown(
        options=["-- select --"] + BIAS_LABELS,
        value="-- select --",
        description="Bias Label:",
        style={"description_width": "100px"},
    )
    
    # Flag checkbox
    flag = widgets.Checkbox(value=False, description="Flag this article (unclear/doesn't fit)")

    # Buttons
    save_btn = widgets.Button(description="Save & Next", button_style="success", icon="check")
    skip_btn = widgets.Button(description="Skip", button_style="warning", icon="forward")
    back_btn = widgets.Button(description="Back", button_style="info", icon="backward")
    
    def on_save(b):
        global current_idx
        if dropdown.value == "-- select --":
            print("Please select a bias label before saving.")
            return
        label = dropdown.value
        if flag.value:
            label = f"FLAG:{label}"
        df.at[idx, "bias_label"] = label
        save_csv()
        current_idx = idx + 1
        show_article(current_idx)

    def on_skip(b):
        global current_idx
        current_idx = idx + 1
        show_article(current_idx)

    def on_back(b):
        global current_idx
        if idx > 0:
            current_idx = idx - 1
            show_article(current_idx)

    save_btn.on_click(on_save)
    skip_btn.on_click(on_skip)
    back_btn.on_click(on_back)

    display(widgets.HBox([dropdown, flag]))
    display(widgets.HBox([back_btn, skip_btn, save_btn]))

# Jump to a specific article number (1-indexed)
def goto(n):
    global current_idx
    current_idx = n - 1
    show_article(current_idx)

show_article(current_idx)

## Utilities
Run `goto(n)` to jump to article number `n`. Run the cell below to see your progress.

In [ ]:
# Progress check
df_check = pd.read_csv(CSV_PATH)
labeled = (df_check["bias_label"].fillna("") != "").sum()
flagged = df_check["bias_label"].fillna("").str.startswith("FLAG:").sum()
print(f"Progress: {labeled}/{len(df_check)} labeled ({labeled/len(df_check)*100:.1f}%)")
print(f"Flagged: {flagged}")
print(f"\nLabel distribution:")
print(df_check[df_check["bias_label"].fillna("") != ""]["bias_label"].value_counts().to_string())